# HVAC Humidity Control using Model Predictive Control (MPC)

This notebook implements a Python version of the MATLAB MPC project for controlling humidity in two interconnected greenhouse zones.

Implemented methods:

- CVXPY-based constrained MPC solver
- SciPy SLSQP solver, similar to MATLAB `fmincon`
- Custom log-barrier solver with gradient descent
- Custom log-barrier solver with Newton updates
- Simulation comparison, control plots, convergence plots, and MSE evaluation

## Mathematical setup

The system is modeled as a linear discrete-time dynamical system:

\[
x_{k+1} = A x_k + B u_k + w_k
\]

where:

- \(x_k \in \mathbb{R}^2\): humidity states for the two greenhouses
- \(u_k \in \mathbb{R}^2\): control actions
- \(w_k\): disturbance/noise
- \(x_{ref}\): target humidity

The MPC objective penalizes:

\[
\sum_{k=1}^{N} \|x_k - x_{ref}\|_Q^2
+ \lambda \sum_{k=0}^{N-1} \|u_k\|_R^2
+ \mu \sum_{k=0}^{N-2} \|u_{k+1} - u_k\|^2
\]

subject to:

\[
0.4 \leq x_k \leq 0.7, \quad -1 \leq u_k \leq 1
\]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize
import pandas as pd

try:
    import cvxpy as cp
    CVXPY_AVAILABLE = True
except ImportError:
    CVXPY_AVAILABLE = False
    print("CVXPY is not installed. Install it with: pip install cvxpy")

In [ ]:
# -----------------------------
# System parameters
# -----------------------------

A = np.array([
    [0.9, 0.1],
    [0.1, 0.9]
], dtype=float)

B = np.array([
    [0.2, 0.1],
    [0.1, 0.2]
], dtype=float)

x_ref = np.array([0.6, 0.5], dtype=float)
u_ref = np.array([0.0, 0.0], dtype=float)

N = 30
lambda_u = 0.1
mu_du = 0.3
T_sim = 17

x0 = np.array([0.45, 0.40], dtype=float)

n = A.shape[0]
m = B.shape[1]

Q = np.eye(n)
R = lambda_u * np.eye(m)

u_ub = np.ones(m)
u_lb = -np.ones(m)

x_min = 0.4
x_max = 0.7

disturbance_traj = np.array([
    [0.001, 0.001, 0.002, -0.03, 0.002, 0.002, 0.002, 0.001, 0.003,
     0.002, 0.001, 0.001, 0.002, 0.002, 0.010, 0.001, 0.002],
    [0.000, 0.001, 0.002, 0.002, 0.001, 0.001, 0.001, 0.000, 0.001,
     0.002, 0.002, 0.001, 0.001, 0.001, 0.002, 0.002, 0.001]
], dtype=float)

print("A =\n", A)
print("B =\n", B)
print("Initial state:", x0)
print("Reference:", x_ref)

In [ ]:
def build_prediction_matrices(A, B, x_initial, N):
    """Build dense prediction matrices:

    X = x_vec + M U

    where X stacks predicted states x_1 ... x_N
    and U stacks controls u_0 ... u_{N-1}.
    """
    n = A.shape[0]
    m = B.shape[1]

    M = np.zeros((N * n, N * m))
    x_vec = np.zeros(N * n)

    for i in range(1, N + 1):
        A_power = np.linalg.matrix_power(A, i)
        x_vec[(i-1)*n:i*n] = A_power @ x_initial

        for j in range(1, i + 1):
            block = np.linalg.matrix_power(A, i - j) @ B
            M[(i-1)*n:i*n, (j-1)*m:j*m] = block

    return x_vec, M


def build_qp_matrices(A, B, Q, R, x_initial, x_ref, u_ref, N, lambda_delta=0.0):
    """Build the dense quadratic program:

    minimize 0.5 U.T H U + q.T U

    The QP is built from state tracking, control effort, and optional control variation penalty.
    """
    n = A.shape[0]
    m = B.shape[1]

    x_vec, M = build_prediction_matrices(A, B, x_initial, N)

    Qblk = np.kron(np.eye(N), Q)
    Rblk = np.kron(np.eye(N), R)

    x_ref_stack = np.tile(x_ref, N)
    u_ref_stack = np.tile(u_ref, N)

    H = M.T @ Qblk @ M + Rblk
    q = M.T @ Qblk @ (x_vec - x_ref_stack) - Rblk @ u_ref_stack

    if lambda_delta > 0:
        D = np.diff(np.eye(N), axis=0)
        S = lambda_delta * np.kron(D.T @ D, np.eye(m))
        H = H + S

    H = 0.5 * (H + H.T)
    return H, q, M, x_vec


def build_linear_constraints_for_barrier(A, B, Q, R, x_initial, x_ref, u_ref, N, u_lb, u_ub):
    """Control-only linear constraints for the barrier methods:

    C U <= d
    """
    H, q, M, x_vec = build_qp_matrices(A, B, Q, R, x_initial, x_ref, u_ref, N)

    C_upper = np.eye(N * m)
    C_lower = -np.eye(N * m)
    d_upper = np.tile(u_ub, N)
    d_lower = -np.tile(u_lb, N)

    C = np.vstack([C_upper, C_lower])
    d = np.concatenate([d_upper, d_lower])

    return H, q, C, d

In [ ]:
def rollout_states(A, B, x_initial, U, N):
    """Roll out states from a stacked control vector U."""
    n = A.shape[0]
    m = B.shape[1]

    X = np.zeros((n, N + 1))
    X[:, 0] = x_initial

    U_mat = U.reshape(N, m)

    for k in range(N):
        X[:, k+1] = A @ X[:, k] + B @ U_mat[k]

    return X


def mpc_cost(U, A, B, x_initial, x_ref, lambda_u, mu_du, N):
    """Full MPC objective used by the SciPy SLSQP solver."""
    m = B.shape[1]
    U_mat = U.reshape(N, m)
    X = rollout_states(A, B, x_initial, U, N)

    cost = 0.0

    for k in range(N):
        cost += np.linalg.norm(X[:, k] - x_ref) ** 2
        cost += lambda_u * np.linalg.norm(U_mat[k]) ** 2

    for k in range(N - 1):
        cost += mu_du * np.linalg.norm(U_mat[k+1] - U_mat[k]) ** 2

    return cost


def state_inequality_constraints(U, A, B, x_initial, N, x_min=0.4, x_max=0.7):
    """Return constraints in scipy-compatible form g(U) >= 0."""
    X = rollout_states(A, B, x_initial, U, N)
    states = X[:, 1:]

    upper_margin = x_max - states.ravel()
    lower_margin = states.ravel() - x_min

    return np.concatenate([upper_margin, lower_margin])

In [ ]:
def solve_mpc_cvxpy(A, B, x_initial, x_ref, lambda_u, mu_du, N, x_min, x_max, u_lb, u_ub):
    """Solve one MPC step using CVXPY."""
    if not CVXPY_AVAILABLE:
        raise ImportError("CVXPY is not installed. Install it with: pip install cvxpy")

    n = A.shape[0]
    m = B.shape[1]

    X = cp.Variable((n, N + 1))
    U = cp.Variable((m, N))

    objective = 0
    constraints = [X[:, 0] == x_initial]

    for k in range(N):
        objective += cp.sum_squares(X[:, k] - x_ref)
        objective += lambda_u * cp.sum_squares(U[:, k])
        constraints.append(X[:, k+1] == A @ X[:, k] + B @ U[:, k])
        constraints.append(X[:, k] >= x_min)
        constraints.append(X[:, k] <= x_max)
        constraints.append(U[:, k] >= u_lb)
        constraints.append(U[:, k] <= u_ub)

    for k in range(N - 1):
        objective += mu_du * cp.sum_squares(U[:, k+1] - U[:, k])

    problem = cp.Problem(cp.Minimize(objective), constraints)

    try:
        problem.solve(solver=cp.OSQP, verbose=False)
    except Exception:
        problem.solve(verbose=False)

    if U.value is None:
        raise RuntimeError("CVXPY failed to solve the MPC problem.")

    return U.value[:, 0]


def run_cvxpy_mpc(A, B, x0, x_ref, lambda_u, mu_du, N, T_sim, disturbance_traj,
                  x_min=0.4, x_max=0.7, u_lb=None, u_ub=None):
    n = A.shape[0]
    m = B.shape[1]

    if u_lb is None:
        u_lb = -np.ones(m)
    if u_ub is None:
        u_ub = np.ones(m)

    x_traj = np.zeros((n, T_sim + 1))
    u_traj = np.zeros((m, T_sim))
    x_current = x0.copy()
    x_traj[:, 0] = x_current

    for t in range(T_sim):
        u_applied = solve_mpc_cvxpy(
            A, B, x_current, x_ref, lambda_u, mu_du, N,
            x_min=x_min, x_max=x_max, u_lb=u_lb, u_ub=u_ub
        )

        x_current = A @ x_current + B @ u_applied + disturbance_traj[:, t]
        x_traj[:, t+1] = x_current
        u_traj[:, t] = u_applied

    return x_traj, u_traj

In [ ]:
def solve_mpc_scipy(A, B, x_initial, x_ref, lambda_u, mu_du, N, x_min, x_max, u_lb, u_ub):
    """Solve one MPC step using scipy.optimize.minimize with SLSQP.

    This is the Python analogue of MATLAB fmincon for this project.
    """
    m = B.shape[1]
    U0 = np.zeros(N * m)

    bounds = [(float(u_lb[i % m]), float(u_ub[i % m])) for i in range(N * m)]

    constraints = {
        "type": "ineq",
        "fun": lambda U: state_inequality_constraints(U, A, B, x_initial, N, x_min, x_max)
    }

    result = minimize(
        fun=lambda U: mpc_cost(U, A, B, x_initial, x_ref, lambda_u, mu_du, N),
        x0=U0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 300, "ftol": 1e-8, "disp": False}
    )

    if not result.success:
        print("SLSQP warning:", result.message)

    return result.x[:m]


def run_scipy_mpc(A, B, x0, x_ref, lambda_u, mu_du, N, T_sim, disturbance_traj,
                  x_min=0.4, x_max=0.7, u_lb=None, u_ub=None):
    n = A.shape[0]
    m = B.shape[1]

    if u_lb is None:
        u_lb = -np.ones(m)
    if u_ub is None:
        u_ub = np.ones(m)

    x_traj = np.zeros((n, T_sim + 1))
    u_traj = np.zeros((m, T_sim))
    x_current = x0.copy()
    x_traj[:, 0] = x_current

    for t in range(T_sim):
        u_applied = solve_mpc_scipy(
            A, B, x_current, x_ref, lambda_u, mu_du, N,
            x_min=x_min, x_max=x_max, u_lb=u_lb, u_ub=u_ub
        )

        x_current = A @ x_current + B @ u_applied + disturbance_traj[:, t]
        x_traj[:, t+1] = x_current
        u_traj[:, t] = u_applied

    return x_traj, u_traj

In [ ]:
def barrier_objective_grad_hess(U, H, q, C, d, tau, compute_hessian=False):
    """Gradient and optional Hessian for:

    0.5 U.T H U + q.T U - tau * sum(log(d - C U))
    """
    slack = d - C @ U

    if np.any(slack <= 0):
        raise ValueError("Infeasible point for log barrier: slack <= 0")

    grad = H @ U + q + tau * (C.T @ (1.0 / slack))

    if not compute_hessian:
        return grad, None

    weighted_C = C / slack[:, None]
    hess = H + tau * (weighted_C.T @ weighted_C)

    return grad, hess


def solve_barrier_gradient(H, q, C, d, epsilon=1e-3, sigma=0.5, alpha=1e-2,
                           max_inner=200, grad_tol=1e-3):
    """Custom log-barrier solver using gradient descent."""
    U = np.zeros(H.shape[0])
    tau = 1.0
    n_constraints = C.shape[0]
    grad_norms = []

    while n_constraints * tau >= epsilon:
        for _ in range(max_inner):
            grad, _ = barrier_objective_grad_hess(U, H, q, C, d, tau)
            norm_grad = np.linalg.norm(grad)
            grad_norms.append(norm_grad)

            if norm_grad < grad_tol:
                break

            step = alpha
            while step > 1e-12:
                U_new = U - step * grad
                if np.all(d - C @ U_new > 0):
                    U = U_new
                    break
                step *= 0.5

            if step <= 1e-12:
                break

        tau *= sigma

    return U, grad_norms


def solve_barrier_newton(H, q, C, d, epsilon=1e-3, sigma=0.5, alpha=1.0,
                         max_inner=100, grad_tol=1e-3):
    """Custom log-barrier solver using Newton updates."""
    U = np.zeros(H.shape[0])
    tau = 1.0
    n_constraints = C.shape[0]
    grad_norms = []

    while n_constraints * tau >= epsilon:
        for _ in range(max_inner):
            grad, hess = barrier_objective_grad_hess(U, H, q, C, d, tau, compute_hessian=True)
            norm_grad = np.linalg.norm(grad)
            grad_norms.append(norm_grad)

            if norm_grad < grad_tol:
                break

            try:
                direction = -np.linalg.solve(hess, grad)
            except np.linalg.LinAlgError:
                direction = -np.linalg.pinv(hess) @ grad

            step = alpha
            while step > 1e-12:
                U_new = U + step * direction
                if np.all(d - C @ U_new > 0):
                    U = U_new
                    break
                step *= 0.5

            if step <= 1e-12:
                break

        tau *= sigma

    return U, grad_norms

In [ ]:
def run_barrier_mpc(A, B, Q, R, x0, x_ref, u_ref, N, T_sim, u_lb, u_ub,
                    disturbance_traj, method="gradient"):
    """Run MPC simulation using custom barrier solvers."""
    n = A.shape[0]
    m = B.shape[1]

    x_traj = np.zeros((n, T_sim + 1))
    u_traj = np.zeros((m, T_sim))
    all_grad_norms = []

    x_current = x0.copy()
    x_traj[:, 0] = x_current

    for t in range(T_sim):
        H, q, C, d = build_linear_constraints_for_barrier(
            A, B, Q, R, x_current, x_ref, u_ref, N, u_lb, u_ub
        )

        if method == "gradient":
            U_opt, grad_norms = solve_barrier_gradient(H, q, C, d)
        elif method == "newton":
            U_opt, grad_norms = solve_barrier_newton(H, q, C, d)
        else:
            raise ValueError("method must be either 'gradient' or 'newton'")

        u_applied = U_opt[:m]
        x_current = A @ x_current + B @ u_applied + disturbance_traj[:, t]

        x_traj[:, t+1] = x_current
        u_traj[:, t] = u_applied
        all_grad_norms.append(grad_norms)

    return x_traj, u_traj, all_grad_norms

In [ ]:
# -----------------------------
# Run simulations
# -----------------------------

results = {}

if CVXPY_AVAILABLE:
    x_cvxpy, u_cvxpy = run_cvxpy_mpc(
        A, B, x0, x_ref, lambda_u, mu_du, N, T_sim, disturbance_traj,
        x_min=x_min, x_max=x_max, u_lb=u_lb, u_ub=u_ub
    )
    results["CVXPY"] = (x_cvxpy, u_cvxpy)
else:
    print("Skipping CVXPY simulation because cvxpy is not installed.")

x_scipy, u_scipy = run_scipy_mpc(
    A, B, x0, x_ref, lambda_u, mu_du, N, T_sim, disturbance_traj,
    x_min=x_min, x_max=x_max, u_lb=u_lb, u_ub=u_ub
)
results["SciPy SLSQP"] = (x_scipy, u_scipy)

x_grad, u_grad, grad_norms = run_barrier_mpc(
    A, B, Q, R, x0, x_ref, u_ref, N, T_sim, u_lb, u_ub,
    disturbance_traj, method="gradient"
)
results["Barrier Gradient"] = (x_grad, u_grad)

x_newton, u_newton, newton_norms = run_barrier_mpc(
    A, B, Q, R, x0, x_ref, u_ref, N, T_sim, u_lb, u_ub,
    disturbance_traj, method="newton"
)
results["Barrier Newton"] = (x_newton, u_newton)

print("Simulations completed:", list(results.keys()))

In [ ]:
def plot_state_comparison(results, x_ref, greenhouse_idx=0):
    t = np.arange(T_sim + 1)

    plt.figure(figsize=(10, 5))
    for name, (x_traj, _) in results.items():
        plt.plot(t, x_traj[greenhouse_idx], label=name)

    plt.axhline(x_ref[greenhouse_idx], linestyle="--", label=f"Reference {greenhouse_idx + 1}")
    plt.axhline(x_min, linestyle=":", label="Min constraint")
    plt.axhline(x_max, linestyle=":", label="Max constraint")

    plt.xlabel("Time step")
    plt.ylabel(f"Humidity - Greenhouse {greenhouse_idx + 1}")
    plt.title(f"Humidity Tracking Comparison - Greenhouse {greenhouse_idx + 1}")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_state_comparison(results, x_ref, greenhouse_idx=0)
plot_state_comparison(results, x_ref, greenhouse_idx=1)

In [ ]:
def plot_control_comparison(results, control_idx=0):
    t = np.arange(T_sim)

    plt.figure(figsize=(10, 5))
    for name, (_, u_traj) in results.items():
        plt.plot(t, u_traj[control_idx], label=name)

    plt.axhline(0, linestyle="--", label="Zero")
    plt.axhline(u_ub[control_idx], linestyle=":", label="Upper bound")
    plt.axhline(u_lb[control_idx], linestyle=":", label="Lower bound")

    plt.xlabel("Time step")
    plt.ylabel(f"Control u_{control_idx + 1}")
    plt.title(f"Control Signal Comparison - u_{control_idx + 1}")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_control_comparison(results, control_idx=0)
plot_control_comparison(results, control_idx=1)

In [ ]:
def tracking_mse(x_traj, x_ref):
    """Mean squared tracking error over the full trajectory."""
    errors = x_traj - x_ref.reshape(-1, 1)
    return np.mean(np.sum(errors ** 2, axis=0))


metrics = []

for name, (x_traj, u_traj) in results.items():
    metrics.append({
        "method": name,
        "tracking_mse": tracking_mse(x_traj, x_ref),
        "mean_abs_control": np.mean(np.abs(u_traj)),
        "max_abs_control": np.max(np.abs(u_traj))
    })

metrics_df = pd.DataFrame(metrics).sort_values("tracking_mse")
metrics_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(metrics_df["method"], metrics_df["tracking_mse"])
plt.ylabel("Tracking MSE")
plt.title("MSE Comparison Across MPC Solvers")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y")
plt.show()

In [ ]:
def flatten_norms(norm_lists):
    values = []
    for norms in norm_lists:
        values.extend(norms)
    return np.array(values)


grad_all = flatten_norms(grad_norms)
newton_all = flatten_norms(newton_norms)

plt.figure(figsize=(10, 5))

if len(grad_all) > 0:
    plt.semilogy(grad_all, label="Barrier Gradient")

if len(newton_all) > 0:
    plt.semilogy(newton_all, label="Barrier Newton")

plt.xlabel("Internal iteration")
plt.ylabel("Gradient norm")
plt.title("Internal Convergence of Custom Barrier Solvers")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Optional: export simulation results for GitHub or reports

export_df = pd.DataFrame({
    "time": np.arange(T_sim + 1)
})

for name, (x_traj, _) in results.items():
    safe_name = name.lower().replace(" ", "_")
    export_df[f"{safe_name}_x1"] = x_traj[0]
    export_df[f"{safe_name}_x2"] = x_traj[1]

export_df.to_csv("hvac_mpc_state_trajectories.csv", index=False)
metrics_df.to_csv("hvac_mpc_metrics.csv", index=False)

print("Saved:")
print("- hvac_mpc_state_trajectories.csv")
print("- hvac_mpc_metrics.csv")